In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t058/task.json'

with open(task_file_path, 'r') as file:
    task_data = json.load(file)

task_data

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge) and was triggered on March 27, 2026, at 16:34:01 (UTC+8). The alert time window is from 16:32:20.924948 to 16:36:01 (UTC+8). The alert entity is an APM operation with the name "inventory::/*". The prompt requests an analysis of the root cause of the issue.

The original code execution output of IPython Kernel is also provided below for reference:

{'task_id': 't058', 'task_version': '1.0', 'alert_event_id': '77541b3b3957d650f151941c2aca69df', 'alert_title': 'inventory错误次数突增', 'alert_trigger_time': '2026-03-27T16:34:01+08:00', 'alert_window': {'start': '2026-03-27T16:32:20.924948+08:00', 'end': '2026-03-27T16:36:01+08:00'}, 'alert_entity': {'entity_id': '4fb1e30f9e19f8b9b15b6bd27a637d7a', 'entity_name': 'inventory::/*', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, 'prompt_text': '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="77541b3b3957d650f151941c2aca69df" trans_id="cdjo4ckfqd49l78hh8ieuvd9u5" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code> 帮我分析下根因。', 'workspace': 'rca-benchmark', 'region_id': 'cn-hongkong', 'available_modalities': ['metrics', 'logs', 'traces', 'events', 'alerts', 'topology'], 'scoring_note': 'Output contract (prediction_schema.json) and fault taxonomy (taxonomy.json) will be published in a follow-up release.'}```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t058/metrics.parquet'
metrics_data = pd.read_parquet(metrics_file_path)

# List all available KPIs and their associated components
kpi_components = metrics_data[['entity_set', 'entity_name', 'metric']].drop_duplicates()

kpi_components

```
Out[2]:
```
The `metrics.parquet` file contains various KPIs across multiple components. These include metrics like `node_ready_status`, `node_memory_usage_rate`, `node_disk_usage_rate`, `node_pod_running_count` for Kubernetes nodes, and `error_rate` for APM instances such as `frontend`, `cart`, and `frontend-proxy`. There are a total of 318 unique combinations of components and KPIs available for analysis.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set                      entity_name                  metric
0          k8s.node                                        node_ready_status
528        k8s.node                                    node_not_ready_status
3432       k8s.node                                   node_memory_usage_rate
5808       k8s.node                                     node_disk_usage_rate
7656       k8s.node                                   node_pod_running_count
...             ...                              ...                     ...
83455  apm.instance        frontend-6df7d557f5-sgfcb              error_rate
83466  apm.instance            cart-67f957cc88-njsn7              error_rate
83477  apm.instance  frontend-proxy-67d64dc4f8-xz2fd              error_rate
83490  apm.instance            cart-67f957cc88-q9w2g              error_rate
83503  apm.instance        frontend-6df7d557f5-nbvwh              error_rate

[318 rows x 3 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Ensure 'time' and 'value' columns are numeric for calculations
metrics_data['time'] = pd.to_numeric(metrics_data['time'], errors='coerce')
metrics_data['value'] = pd.to_numeric(metrics_data['value'], errors='coerce')

# Group by component-KPI combination and calculate the global P95 threshold
p95_thresholds = metrics_data.groupby(['entity_set', 'entity_name', 'metric'])['value'].quantile(0.95).reset_index()
p95_thresholds.rename(columns={'value': 'p95_threshold'}, inplace=True)

p95_thresholds

```
Out[3]:
```
The global P95 thresholds for each component-KPI combination have been calculated across the entire dataset. For example, the `avg_request_latency_seconds` for the `accounting` component has a P95 threshold of 0.05192, while its `request_count` has a threshold of 1874.25. Similarly, the `node_memory_usage_rate` for Kubernetes nodes has a P95 threshold of 92.816244. These thresholds will be used to identify anomalies in the next step. There are a total of 318 component-KPI combinations with their respective thresholds.

The original code execution output of IPython Kernel is also provided below for reference:

entity_set            entity_name                       metric  p95_threshold
0    apm.instance             accounting  avg_request_latency_seconds       0.051920
1    apm.instance             accounting                request_count    1874.250000
2    apm.instance                     ad  avg_request_latency_seconds       0.001186
3    apm.instance                     ad                request_count    4595.450000
4    apm.instance  cart-67f957cc88-njsn7  avg_request_latency_seconds       0.009103
..            ...                    ...                          ...            ...
313      k8s.node                                node_disk_usage_rate      32.188594
314      k8s.node                              node_memory_usage_rate      92.816244
315      k8s.node                               node_not_ready_status       0.000000
316      k8s.node                              node_pod_running_count     773.000000
317      k8s.node                                   node_ready_status       1.000000

[318 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Convert alert time window to UTC+8 timestamps in microseconds
timezone = pytz.timezone('Asia/Shanghai')
alert_start = int(datetime.strptime('2026-03-27T16:32:20.924948+08:00', '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
alert_end = int(datetime.strptime('2026-03-27T16:36:01+08:00', '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter metrics data within the alert time window
filtered_metrics = metrics_data[(metrics_data['time'] >= alert_start) & (metrics_data['time'] <= alert_end)]

# Merge filtered metrics with P95 thresholds
merged_data = filtered_metrics.merge(p95_thresholds, on=['entity_set', 'entity_name', 'metric'], how='left')

# Identify anomalies where the value exceeds the P95 threshold
anomalies = merged_data[merged_data['value'] > merged_data['p95_threshold']]

# Focus on the 'inventory' service and other possible root cause components
inventory_anomalies = anomalies[anomalies['entity_name'].str.contains('inventory', case=False)]
inventory_anomalies

```
Out[4]:
```
During the alert time window, several anomalies were detected in the `inventory` service. Key findings include:

1. Elevated `avg_request_latency_seconds` across multiple instances and operations, exceeding the P95 thresholds.
2. High JVM garbage collection time (`arms_jvm_gc_seconds_delta`) and memory usage (`arms_jvm_mem_committed_bytes`), both surpassing their respective thresholds.
3. Increased service-level latency and error counts, with values significantly above the P95 thresholds.
4. Exceptionally high `error_rate` for specific instances, such as `inventory-887d8db89-7v4s5`.

These anomalies suggest performance degradation and error spikes in the `inventory` service during the alert period. Further investigation is needed to pinpoint the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

time domain          entity_set                         entity_id                               entity_name                        metric         value                            metric_set_id    service  p95_threshold
3241  1774600400000000    apm       apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}   avg_request_latency_seconds  1.107339e-02  apm@metric_set@apm.metric.apm.operation       None   8.447195e-03
3263  1774600370000000    apm       apm.operation  4fb1e30f9e19f8b9b15b6bd27a637d7a                             inventory::/*   avg_request_latency_seconds  5.085106e-03  apm@metric_set@apm.metric.apm.operation       None   3.875162e-03
3770  1774600370000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory   avg_request_latency_seconds  1.832051e-02   apm@metric_set@apm.metric.apm.instance       None   9.633365e-03
3771  1774600400000000    apm        apm.instance  33c16dd0c27f7e1a60ad4bc3adcb47ba                                 inventory   avg_request_latency_seconds  2.029557e-02   apm@metric_set@apm.metric.apm.instance       None   9.633365e-03
3812  1774600370000000    apm        apm.instance  2da48118270a22f4de7ac642102def69                 inventory-887d8db89-7v4s5   avg_request_latency_seconds  1.986188e-02   apm@metric_set@apm.metric.apm.instance       None   1.508050e-02
3847  1774600370000000    apm        apm.instance  c6cbf1b49bcdb7d33859d04c79b97bf8                 inventory-87855b9b9-bjrcs   avg_request_latency_seconds  1.650794e-02   apm@metric_set@apm.metric.apm.instance       None   1.162325e-02
3848  1774600400000000    apm        apm.instance  c6cbf1b49bcdb7d33859d04c79b97bf8                 inventory-87855b9b9-bjrcs   avg_request_latency_seconds  2.136898e-02   apm@metric_set@apm.metric.apm.instance       None   1.162325e-02
3924  1774600460000000    apm      apm.metric.jvm                                                                   inventory     arms_jvm_gc_seconds_total  6.484760e+02                                     None  inventory   6.477945e+02
3925  1774600520000000    apm      apm.metric.jvm                                                                   inventory     arms_jvm_gc_seconds_total  6.484760e+02                                     None  inventory   6.477945e+02
3929  1774600400000000    apm      apm.metric.jvm                                                                   inventory     arms_jvm_gc_seconds_delta  6.434000e+00                                     None  inventory   1.907000e+00
3939  1774600460000000    apm      apm.metric.jvm                                                                   inventory  arms_jvm_mem_committed_bytes  1.220137e+09                                     None  inventory   1.220126e+09
3940  1774600520000000    apm      apm.metric.jvm                                                                   inventory  arms_jvm_mem_committed_bytes  1.220181e+09                                     None  inventory   1.220126e+09
4037  1774600370000000    apm  apm.service.legacy                                                                   inventory                       latency  1.805397e-02                                     None       None   8.175712e-03
4038  1774600370000000    apm  apm.service.legacy                                                                   inventory                           mem  1.570599e+09                                     None       None   1.551813e+09
4112  1774600400000000    apm  apm.service.legacy                                                                   inventory                       latency  1.859558e-02                                     None       None   8.175712e-03
4186  1774600430000000    apm  apm.service.legacy                                                                   inventory                         error  5.730000e+02                                     None       None   5.208500e+02
4261  1774600460000000    apm  apm.service.legacy                                                                   inventory                         error  5.550000e+02                                     None       None   5.208500e+02
4411  1774600520000000    apm  apm.service.legacy                                                                   inventory                         error  5.280000e+02                                     None       None   5.208500e+02
4486  1774600550000000    apm  apm.service.legacy                                                                   inventory                         error  5.780000e+02                                     None       None   5.208500e+02
4488  1774600550000000    apm  apm.service.legacy                                                                   inventory                           mem  1.551955e+09                                     None       None   1.551813e+09
4562  1774600550000000    apm        apm.instance  2da48118270a22f4de7ac642102def69                 inventory-887d8db89-7v4s5                    error_rate  9.146919e-01   apm@metric_set@apm.metric.apm.instance       None   9.142838e-01```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t058/traces.parquet'
traces_data = pd.read_parquet(traces_file_path)

# Ensure 'startTime' and 'endTime' are numeric for filtering
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')

# Filter traces within the alert time window
filtered_traces = traces_data[
    (traces_data['startTime'] / 1_000_000 >= alert_start) & 
    (traces_data['startTime'] / 1_000_000 <= alert_end)
]

# Focus on traces involving the 'inventory' service
inventory_traces = filtered_traces[filtered_traces['serviceName'].str.contains('inventory', case=False, na=False)]

inventory_traces

```
Out[5]:
```
No traces involving the `inventory` service were found within the alert time window (16:32:20.924948 to 16:36:01 UTC+8). This could indicate that the issue did not propagate through distributed traces or that the relevant traces were not captured during this period. Further investigation using other telemetry data, such as logs or events, may be necessary to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, ip]
Index: []

[0 rows x 21 columns]```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t058/logs.parquet'
logs_data = pd.read_parquet(logs_file_path)

# Ensure '_time_' is in datetime format for filtering
logs_data['_time_'] = pd.to_datetime(logs_data['_time_'], errors='coerce')

# Filter logs within the alert time window
filtered_logs = logs_data[
    (logs_data['_time_'] >= datetime.fromtimestamp(alert_start / 1_000_000, timezone)) &
    (logs_data['_time_'] <= datetime.fromtimestamp(alert_end / 1_000_000, timezone))
]

# Focus on logs related to the 'inventory' service and look for error messages
inventory_logs = filtered_logs[
    filtered_logs['_container_name_'].str.contains('inventory', case=False, na=False) |
    filtered_logs['content'].str.contains('inventory', case=False, na=False)
]

# Further filter for error messages
error_logs = inventory_logs[inventory_logs['content'].str.contains('error', case=False, na=False)]

error_logs

```
Out[6]:
```
No error logs related to the `inventory` service were found within the alert time window (16:32:20.924948 to 16:36:01 UTC+8). This suggests that either the issue did not generate error logs or the relevant logs were not captured during this period. Further investigation using other telemetry data, such as events or additional metrics, may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns]```
```

In [ ]:
# In[7]:

# Inspect the columns in the events.parquet file
events_data.columns

```
Out[7]:
```
The `events.parquet` file contains columns such as `eventId`, `hostname`, `level`, `pod_id`, `pod_name`, `clusterId`, and `clusterName`. These columns can be used to filter and analyze events related to the `inventory` service or its associated pods and nodes. Next, we will filter the events within the alert time window and focus on those related to the `inventory` service.

The original code execution output of IPython Kernel is also provided below for reference:

Index(['eventId', 'hostname', 'level', 'pod_id', 'pod_name', 'clusterId', 'clusterName', '__topic__'], dtype='object')```
```

In [ ]:
# In[8]:

# Ensure the 'eventId' column is parsed to extract timestamps for filtering
events_data['eventId'] = events_data['eventId'].apply(json.loads)

# Extract the timestamp from the 'eventId' JSON payload and convert to datetime
events_data['event_time'] = pd.to_datetime(
    events_data['eventId'].apply(lambda x: x.get('metadata', {}).get('creationTimestamp', None)),
    errors='coerce'
)

# Filter events within the alert time window
filtered_events = events_data[
    (events_data['event_time'] >= datetime.fromtimestamp(alert_start / 1_000_000, timezone)) &
    (events_data['event_time'] <= datetime.fromtimestamp(alert_end / 1_000_000, timezone))
]

# Focus on events related to the 'inventory' service, its associated pods, or nodes
inventory_events = filtered_events[
    filtered_events['pod_name'].str.contains('inventory', case=False, na=False) |
    filtered_events['hostname'].str.contains('inventory', case=False, na=False)
]

inventory_events

```
Out[8]:
```
No events related to the `inventory` service, its associated pods, or nodes were found within the alert time window (16:32:20.924948 to 16:36:01 UTC+8). This indicates that no Kubernetes-level issues, such as pod crashes or node problems, were recorded for the `inventory` service during this period. Further investigation may require revisiting metrics or other telemetry data.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [eventId, hostname, level, pod_id, pod_name, clusterId, clusterName, __topic__, event_time]
Index: []```
```